# Session 36 Solutions: Preparing Data for Models

Data Science with Python, Level 3, Stage 6: Applied ML

Try each exercise yourself before reading its solution. There is usually more than one correct answer; compare your approach with this one.

## Setup

In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import train_test_split, cross_val_score

DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"
churn = pd.read_csv(DATA_URL + "telecom_churn.csv")
X = churn.drop(columns=["customer_id", "churn"])
y = (churn["churn"] == "Yes").astype(int)

## In-Class Exercises

### In-Class Exercise 1: Payment Dummies

In [2]:
enc = OneHotEncoder(sparse_output=False)
pm = pd.DataFrame(enc.fit_transform(X[["payment_method"]]), columns=enc.get_feature_names_out())
pm.sum()

payment_method_Bank transfer       483.0
payment_method_Card                583.0
payment_method_Cash                275.0
payment_method_Electronic check    659.0
dtype: float64

### In-Class Exercise 2: Survey Gaps

In [3]:
survey = pd.read_csv(DATA_URL + "messy_survey.csv")
print(survey.isna().sum())
survey[["monthly_income", "hours_online"]] = SimpleImputer(strategy="median").fit_transform(survey[["monthly_income", "hours_online"]])
survey[["city"]] = SimpleImputer(strategy="most_frequent").fit_transform(survey[["city"]])
survey[["monthly_income", "hours_online", "city"]].isna().sum()

respondent        0
name              0
age               0
city              4
monthly_income    9
hours_online      6
satisfaction      3
signup_date       0
dtype: int64


monthly_income    0
hours_online      0
city              0
dtype: int64

### In-Class Exercise 3: Salary Preprocessor

In [4]:
emp = pd.read_csv(DATA_URL + "employees.csv")
pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["department", "city"]),
    ("lvl", OrdinalEncoder(categories=[["Junior", "Mid", "Senior", "Lead"]]), ["level"]),
    ("num", StandardScaler(), ["years_experience", "performance"]),
])
print(pre.fit_transform(emp).shape)
print(pre.get_feature_names_out())

(400, 13)
['cat__department_Data' 'cat__department_Engineering'
 'cat__department_Finance' 'cat__department_Marketing'
 'cat__department_Sales' 'cat__department_Support' 'cat__city_Amman'
 'cat__city_Beirut' 'cat__city_Cairo' 'cat__city_Dubai' 'lvl__level'
 'num__years_experience' 'num__performance']


### In-Class Exercise 4: Does It Help?

In [5]:
num_cols = ["tenure_months", "monthly_charges", "total_charges", "support_calls"]
def churn_model(cat_cols, bool_cols):
    pre = ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),
        ("cat", make_pipeline(SimpleImputer(strategy="constant", fill_value="None"), OneHotEncoder(handle_unknown="ignore")), cat_cols),
        ("bool", "passthrough", bool_cols),
    ])
    return make_pipeline(pre, LogisticRegression(max_iter=1000))
full = churn_model(["gender", "contract", "internet", "payment_method"], ["senior", "paperless_billing"])
small = churn_model(["contract", "internet", "payment_method"], ["senior"])
for name, m in [("full", full), ("small", small)]:
    print(name, cross_val_score(m, X, y, cv=5, scoring="roc_auc").mean().round(3))

full 0.787
small 0.788


## Part A: Basic

### Exercise 1: get_dummies versus OneHotEncoder

In [6]:
print(list(pd.get_dummies(X[["gender", "contract"]]).columns))
enc = OneHotEncoder().fit(X[["gender", "contract"]])
print(list(enc.get_feature_names_out()))
# The encoder remembers the training categories, so new data always gets the same columns.

['gender_Female', 'gender_Male', 'contract_Month-to-month', 'contract_One year', 'contract_Two year']
['gender_Female', 'gender_Male', 'contract_Month-to-month', 'contract_One year', 'contract_Two year']


### Exercise 2: Imputation Strategies

In [7]:
for s in ["mean", "median", "most_frequent"]:
    print(s, SimpleImputer(strategy=s).fit(X[["total_charges"]]).statistics_.round(2))
# total_charges is right-skewed: a few long-term customers pull the mean up.

mean [1876.97]
median [1556.47]
most_frequent [585.85]


## Part B: Intermediate

### Exercise 3: Salary Model

In [8]:
emp = pd.read_csv(DATA_URL + "employees.csv")
pre = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["department", "city"]),
    ("lvl", OrdinalEncoder(categories=[["Junior", "Mid", "Senior", "Lead"]]), ["level"]),
    ("num", StandardScaler(), ["years_experience", "performance"]),
    ("bool", "passthrough", ["remote"]),
])
model = make_pipeline(pre, LinearRegression())
cols = ["department", "city", "level", "years_experience", "performance", "remote"]
print(cross_val_score(model, emp[cols], emp["monthly_salary"], cv=5, scoring="r2").mean().round(3))

0.92


### Exercise 4: Ordinal or One-Hot?

In [9]:
emp = pd.read_csv(DATA_URL + "employees.csv")
cols = ["department", "city", "level", "years_experience", "performance", "remote"]
def salary_model(level_step):
    pre = ColumnTransformer([
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["department", "city"]),
        ("lvl", level_step, ["level"]),
        ("num", StandardScaler(), ["years_experience", "performance"]),
        ("bool", "passthrough", ["remote"]),
    ])
    return make_pipeline(pre, LinearRegression())
for name, step in [("ordinal", OrdinalEncoder(categories=[["Junior", "Mid", "Senior", "Lead"]])), ("one-hot", OneHotEncoder())]:
    print(name, cross_val_score(salary_model(step), emp[cols], emp["monthly_salary"], cv=5, scoring="r2").mean().round(3))
# One-hot lets each level have its own salary jump; ordinal forces equal steps between levels.

ordinal 0.92
one-hot 0.918


## Part C: Challenge

### Exercise 5: A Reusable Function

Selecting both `object` and `string` makes the function work in Colab and in newer pandas versions, which store text differently.

In [10]:
def make_preprocessor(df, target):
    feats = df.drop(columns=[target])
    num = list(feats.select_dtypes("number").columns)
    txt = list(feats.select_dtypes(["object", "string"]).columns)
    bools = list(feats.select_dtypes("bool").columns)
    return ColumnTransformer([
        ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num),
        ("cat", make_pipeline(SimpleImputer(strategy="constant", fill_value="None"), OneHotEncoder(handle_unknown="ignore")), txt),
        ("bool", "passthrough", bools),
    ])

data = churn.drop(columns=["customer_id"])
pre = make_preprocessor(data, "churn")
print(pre.fit_transform(data.drop(columns=["churn"])).shape)
flats = pd.read_csv(DATA_URL + "apartments.csv").drop(columns=["apartment_id"])
pre2 = make_preprocessor(flats, "price_k")
print(pre2.fit_transform(flats.drop(columns=["price_k"])).shape)

(2000, 18)
(800, 12)


## Part D: Cumulative Project: Customer Churn

### Exercise 6: Customer Churn: Step 1, The Preprocessor

In [11]:
churn = pd.read_csv(DATA_URL + "telecom_churn.csv")
y = (churn["churn"] == "Yes").astype(int)
X = churn.drop(columns=["customer_id", "churn"])
X_dev, X_final, y_dev, y_final = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

num_cols = ["tenure_months", "monthly_charges", "total_charges", "support_calls"]
cat_cols = ["gender", "contract", "internet", "payment_method"]
bool_cols = ["senior", "paperless_billing"]
preprocess = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num_cols),
    ("cat", make_pipeline(SimpleImputer(strategy="constant", fill_value="None"), OneHotEncoder(handle_unknown="ignore")), cat_cols),
    ("bool", "passthrough", bool_cols),
])
model = make_pipeline(preprocess, LogisticRegression(max_iter=1000))
print("CV AUC:", cross_val_score(model, X_dev, y_dev, cv=5, scoring="roc_auc").mean().round(3))
# Missing internet means the customer has no internet service, so it becomes its own category "None".
# Dropping those rows would remove 20 percent of the customers and a real group.

CV AUC: 0.786


## Part E: Quiz Answers

| Question | Answer | Explanation |
| --- | --- | --- |
| 1 | B | Text must be encoded first. |
| 2 | B | Cities have no natural order. |
| 3 | B | New categories become all-zero rows. |
| 4 | B | Learning from the test set leaks information. |
| 5 | B | Each group gets its own transformer. |